**`04_show_geographic_groupings`**

Tutorial for the different ways one town's parcels can be grouped:
the town itself (`admin4_id`), Census tracts, block groups, blocks,
and ZIP Code Tabulation Areas (`zcta5_id`). All five columns are
assigned to parcels by the `link_geographic_ids` harmonize step; this
notebook only reads already-processed output -- it assumes
`US_parcel-spine-2026` has already been harmonized for the chosen
county.

Each map is a parcel-level choropleth -- every parcel is drawn and
colored by its own id value, with no boundary lines between parcels
and no reference-tile geometry overlaid. The point is to make the
resolution of the measure itself unmistakable: this is a parcel-level
attribute, not a smaller number of large regions.

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import hsv_to_rgb

import openplaces as op

# Pick a processed county and its largest town

In [ ]:
COUNTY_ID = 'US-NC-BRU'  # Brunswick County, NC -- swap for any other processed county

parcels = op.get_entities('US_parcel-spine-2026', COUNTY_ID, geom=True)
TOWN_ID = parcels['admin4_id'].value_counts().idxmax()
town = parcels[parcels['admin4_id'] == TOWN_ID].copy()
print(f'{TOWN_ID}: {len(town):,} parcels')

# Five groupings, one parcel-level map each

Every distinct id value gets its own randomly-drawn pastel color (high
value, muted saturation) -- not a small fixed palette that would repeat
colors across many groups and understate how fine-grained a measure
actually is. When a column has more distinct values than fit in a
legend, only the 7 most frequent (by parcel count) are labeled; every
parcel still keeps its own group's color on the map itself.

In [ ]:
def pastel_colors(n, seed=0):
    """n random pastel RGB colors, one per distinct category value.

    A small fixed qualitative colormap (e.g. tab10) would cycle and
    repeat once a column has more distinct values than colors, making a
    fine-grained measure look coarser than it is. Drawing hue freely
    while keeping saturation muted and value high gives each category
    a distinct, pastel color instead.
    """
    rng = np.random.default_rng(seed)
    hsv = np.column_stack(
        [
            rng.uniform(0, 1, n),
            rng.uniform(0.3, 0.55, n),
            rng.uniform(0.85, 1.0, n),
        ]
    )
    return hsv_to_rgb(hsv)


def plot_parcel_grouping(ax, gdf, column, title, legend_max=7, seed=0):
    cat = pd.Categorical(gdf[column])
    valid = cat.codes >= 0
    n = len(cat.categories)
    colors = pastel_colors(n, seed=seed)

    # No edgecolor at all: parcel-to-parcel boundaries would visually
    # read as a grid overlay rather than a smooth, parcel-resolution
    # choropleth.
    gdf.loc[valid].plot(ax=ax, color=colors[cat.codes[valid]], edgecolor='none')

    counts = gdf.loc[valid, column].value_counts()
    color_by_value = dict(zip(cat.categories, colors, strict=True))
    handles = [
        mpatches.Patch(facecolor=color_by_value[v], edgecolor='none', label=str(v))
        for v in counts.index[:legend_max]
    ]
    if n > legend_max:
        # No swatch for this entry (facecolor/edgecolor both 'none') --
        # it's a count, not another category to look for on the map.
        handles.append(
            mpatches.Patch(
                facecolor='none', edgecolor='none', label=f'and {n - legend_max} more'
            )
        )
    ax.legend(
        handles=handles,
        fontsize='x-small',
        loc='upper center',
        bbox_to_anchor=(0.5, -0.02),
        ncol=2,
        frameon=False,
    )
    ax.set_title(title)
    ax.set_aspect('equal')
    ax.axis('off')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 11))
plot_parcel_grouping(axes[0, 0], town, 'admin4_id', 'Town (admin4)')
plot_parcel_grouping(axes[0, 1], town, 'census_tract_id', 'Census tract')
plot_parcel_grouping(axes[0, 2], town, 'census_blockgroup_id', 'Block group')
plot_parcel_grouping(axes[1, 0], town, 'census_block_id', 'Block')
plot_parcel_grouping(axes[1, 1], town, 'zcta5_id', 'ZCTA (ZIP)')
axes[1, 2].axis('off')
fig.suptitle(f'{TOWN_ID}: {len(town):,} parcels, grouped five ways')
fig.subplots_adjust(hspace=0.5, wspace=0.15)